__Este notebook demonstra como realizar Transfer Learning utilizando o modelo MobileNetV2__

## Recursos necessários
- **Python 3.10**: O interpretador base da linguagem precisa estar nas versões de  3.7 a 3.10 para ser compatível com o TensorFlow.
- **TensorFlow**: Responsável principal pela feitura do processo de treinamento.
- **NumPy**: Responsável pelos cálculos de adequação para visualização dos objetos de exemplo.
- **Matplotlib**: Responsável pela exibição de objetos de exemplo.


In [1]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing import image
import numpy as np
import matplotlib.pyplot as plt



## Carregamento e Configuração do Modelo Base

1. **Utilizando o modelo pré-treinado MobileNetV2, Inception e ResNet50**:
    - **MobileNetV2** é uma arquitetura de rede neural leve e eficiente, pré-treinada em um grande conjunto de dados 

In [2]:


# Carregar o modelo pré-treinado, excluindo a última camada
base_model = MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')


 

2. **Congelando as camadas do modelo base**:
    - Congelar as camadas significa impedir que os pesos dessas camadas sejam atualizados durante o treinamento. Isso é útil quando se deseja manter o conhecimento pré-treinado e apenas ajustar as camadas superiores para a tarefa específica.


In [3]:

# Congelar as camadas do modelo base
base_model.trainable = False



3. **Adicionando camadas ao modelo**:
    - Adicionar camadas ao modelo envolve anexar novas camadas (como camadas densas ou de dropout) ao modelo base para adaptar a rede à nova tarefa. Essas camadas serão treinadas a partir do zero.


In [4]:

model = models.Sequential([
    base_model,  
    layers.GlobalAveragePooling2D(),  
    layers.Dense(128, activation='relu'), 
    layers.Dropout(0.3),  
    layers.Dense(5, activation='softmax') 
])



4. **Compilando o modelo**:
    - Compilar o modelo é o processo de configurar a função de perda, o otimizador e as métricas que serão usadas durante o treinamento. Isso prepara o modelo para o processo de treinamento.


In [5]:

# Compilar o modelo
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)



5. **Exibindo o resumo do modelo**:
    - Exibir o resumo do modelo (`model.summary()`) fornece uma visão geral da arquitetura do modelo, incluindo o número de camadas, tipos de camadas, formas de saída e o número de parâmetros treináveis e não treináveis.


In [ ]:

# Exemplo de resumo do modelo
model.summary()



6. **Criando geradores de dados para treinamento e validação**:
    - Geradores de dados são utilizados para carregar e pré-processar os dados em tempo real durante o treinamento. Eles aplicam transformações como redimensionamento e normalização, e dividem os dados em conjuntos de treinamento e validação.


In [ ]:
# Adição dos novos dados
# Criar geradores de dados para treinamento e validação
train_datagen = tf.keras.preprocessing.image.ImageDataGenerator(rescale=1./255, validation_split=0.2)

# Carregar os dados de treinamento e validação a partir da pasta 'dados'
train_dataset = train_datagen.flow_from_directory(
    'Dados',
    target_size=(224, 224),
    batch_size=32,
    class_mode='sparse',
    subset='training'
)

val_dataset = train_datagen.flow_from_directory(
    'Dados',
    target_size=(224, 224),
    batch_size=32,
    class_mode='sparse',
    subset='validation'
)




7. **Carregar os dados de treinamento e validação**:
    - Carregar os dados envolve utilizar os geradores de dados para ler imagens de um diretório, redimensioná-las para um tamanho específico, agrupar as imagens em lotes e rotulá-las para treinamento e validação.




In [ ]:


#definir imagense e etiquetas
imagens = train_dataset[0][0]
etiquetas = train_dataset[0][1]

# Nomear as classes do conjunto de dados com base no nome de pasta
class_names = list(train_dataset.class_indices.keys())
print(class_names)
# Fazer um loader de imagens e etiquetas do conjunto de treinamento
train_loader = tf.data.Dataset.from_tensor_slices((imagens, etiquetas))


# Visualizar algumas imagens do batch
fig, axes = plt.subplots(1, 6, figsize=(12, 3))
for i in range(6):
    axes[i].imshow(imagens[i].squeeze(), cmap='Greys')  # Remove o canal de cor com `.squeeze()`
    axes[i].set_title(f"{class_names[int(etiquetas[i].item())]}")
    axes[i].axis('off')
plt.show()

7.1 **Função de ajuste de modelo (fit)**

In [ ]:

# Treinar o modelo 
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10
    
)


7.2 **Avaliação do modelo**

In [ ]:
# Avaliar o modelo
model.evaluate(val_dataset)

# mostrar a acurácia e a perda  do modelo
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Acurácia do Modelo')
plt.ylabel('Acurácia')
plt.xlabel('Época')
plt.legend(['Treinamento', 'Validação'], loc='upper left')
plt.show()

7.3 **Teste de predição**

In [ ]:
# Visualizar identificação de objetos 

# Carregar uma imagem de teste
img_path = 'validacao/capivara/00000003.jpg'
img = image.load_img(img_path, target_size=(224, 224))
img_array = image.img_to_array(img)
img_array = preprocess_input(img_array)
img_array = np.expand_dims(img_array, axis=0)

# Fazer a predição
pred = model.predict(img_array)
pred_class = np.argmax(pred)
pred_prob = pred[0, pred_class]
class_names = ['capivara', 'porquinho_da_india']
# Mostrar a imagem e a predição
plt.imshow(img)
plt.axis('off')
plt.title(f'Predição: {class_names[pred_class]} ({pred_prob*100:.2f}%)')
plt.show()